## 12 - Pivot Tables & Reshaping

Reshaping lets you look at the same data from different angles.

### pivot_table() - Excel-style aggregation
```python
df.pivot_table(
    values='score',        # which column to aggregate
    index='city',          # becomes the row labels
    columns='gender',      # becomes the column labels
    aggfunc='mean',        # aggregation function
    fill_value=0           # replace NaN
)
```

### melt() - wide to long format
```python
# Wide: one column per subject
# Long: one row per subject per student
df.melt(id_vars=['name'],
        value_vars=['maths','science','english'],
        var_name='subject',
        value_name='score')
```

### stack() and unstack()
```python
df.stack()    # move column labels to row index (wide → tall)
df.unstack()  # move row index level to columns (tall → wide)
```

### crosstab() - frequency table
```python
pd.crosstab(df['city'], df['grade'])          # counts
pd.crosstab(df['city'], df['grade'],
            normalize='index')                # row percentages
```


In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('students_enriched.csv')

In [3]:
score_cols = ['maths','science','english','compSci']
for col in score_cols: df[col] = df[col].fillna(df[col].mean())

In [4]:
# pivot table

print('Average maths by study category and gender:')

pivot = df.pivot_table(
    values='maths',
    index='study_category',
    columns='gender',
    aggfunc='mean',
    margins=True,          # adds row/col totals
    margins_name='Overall'
).round(2)

print(pivot)

Average maths by study category and gender:
gender          Female   Male  Overall
study_category                        
High             58.62  67.98    65.51
Low              66.36  60.90    64.68
Medium           69.09  59.06    64.08
Overall          65.74  63.96    64.78


In [5]:
# Multi-value pivot
multi_pivot = df.pivot_table(
    values=['maths','english'],
    index='study_category',
    aggfunc={'maths':'mean','english':'mean'}
).round(2)

print('Multi-subject pivot:')
print(multi_pivot)

Multi-subject pivot:
                english  maths
study_category                
High              72.54  65.51
Low               72.55  64.68
Medium            70.74  64.08


In [6]:
# melt() - wide to long 
df_long = df.melt(
    id_vars=['student_id','name','study_category','gender'],
    value_vars=score_cols,
    var_name='subject',
    value_name='score'
)

print(f'Wide shape: {df.shape}  →  Long shape: {df_long.shape}')
print(df_long.head(8))

Wide shape: (50, 19)  →  Long shape: (200, 6)
  student_id    name study_category  gender subject  score
0       S001   Aarav            Low  Female   maths   52.9
1       S002   Aanya         Medium    Male   maths   57.5
2       S003   Aditi         Medium  Female   maths   78.7
3       S004   Arjun         Medium  Female   maths   69.9
4       S005  Bhavna           High  Female   maths   57.1
5       S006  Chirag         Medium    Male   maths   72.7
6       S007   Deepa         Medium  Female   maths   66.5
7       S008   Dhruv         Medium  Female   maths   79.5


In [7]:
# Aggregation on long format 
print('Average per subject per study category (from long format):')

print(
    df_long.groupby(['study_category','subject'])['score']
    .mean()
    .round(2)
    .unstack()
)

Average per subject per study category (from long format):
subject         compSci  english  maths  science
study_category                                  
High              70.41    72.54  65.51    74.01
Low               76.32    72.55  64.68    66.68
Medium            66.45    70.74  64.08    67.92


In [8]:
# crosstab
print('Grade distribution by study category:')

ct = pd.crosstab(df['study_category'], df['grade'])
print(ct)

Grade distribution by study category:
grade           A  B   C  D
study_category             
High            1  7  11  0
Low             0  5   8  0
Medium          0  5  12  1


In [9]:
print('Grade distribution by study category (row %):')

ct_pct = pd.crosstab(
    df['study_category'],
    df['grade'],
    normalize='index'
).round(3) * 100

print(ct_pct)

Grade distribution by study category (row %):
grade             A     B     C    D
study_category                      
High            5.3  36.8  57.9  0.0
Low             0.0  38.5  61.5  0.0
Medium          0.0  27.8  66.7  5.6
